# FYP2 模型训练 — AutoDL A100-PCIE-40GB

**使用说明**：从头到尾顺序执行每个 Cell。

- 数据放在 `/root/autodl-tmp/CCV2.v2i.yolov81024/`
- 项目代码放在 `/root/autodl-tmp/fyp_withoutRebar/`
- 训练输出自动写入 `/root/autodl-tmp/fyp_withoutRebar/runs/train/`

## 0. 环境检查

In [ ]:
import subprocess, sys, os

print("Python:", sys.version)
print("Working dir:", os.getcwd())

# GPU check
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print("GPU:", r.stdout.strip())

# Disk check
for path in ["/root/autodl-tmp"]:
    if os.path.exists(path):
        s = os.statvfs(path)
        free_gb = s.f_frsize * s.f_bavail / 1e9
        print(f"{path}: {free_gb:.0f} GB free")

## 1. 安装依赖

In [ ]:
# AutoDL 镜像一般已有 PyTorch + CUDA，先降级 numpy 再补依赖
!pip install "numpy<2" ultralytics opencv-python-headless scikit-image flask -q

# 验证
import torch, ultralytics
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
print(f"Ultralytics {ultralytics.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 检查项目代码 & 数据集

> 在执行前，请确保已用 AutoDL 的\"文件上传\"或 `scp` 把以下内容上传到 `/root/autodl-tmp/`：
> - `fyp_withoutRebar/`（项目代码，或上传整个 zip 后解压）
> - `CCV2.v2i.yolov81024/`（数据集）
>
> AutoDL 的 Jupyter 自带文件上传功能，在左侧文件浏览器拖拽即可。

In [ ]:
import os

BASE = "/root/autodl-tmp"
PROJECT = os.path.join(BASE, "fyp_withoutRebar")
DATA = os.path.join(BASE, "CCV2.v2i.yolov81024")

print("Project dir:", "OK" if os.path.isdir(PROJECT) else "MISSING — please upload fyp_withoutRebar/")
print("Dataset dir:", "OK" if os.path.isdir(DATA) else "MISSING — please upload CCV2.v2i.yolov81024/")

# Check key files
for f in ["train.py", "modules/skeleton.py", "requirements.txt"]:
    p = os.path.join(PROJECT, f)
    print(f"  {f}: {'OK' if os.path.exists(p) else 'MISSING'}")

for f in ["data.yaml", "train/images", "test/images"]:
    p = os.path.join(DATA, f)
    print(f"  data/{f}: {'OK' if os.path.exists(p) else 'MISSING'}")

## 3. 数据集审计（确认数据完整）

In [ ]:
import sys; sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

from pathlib import Path
from dataset_audit import audit_dataset
import json

data_yaml = Path(os.path.join(DATA, "data.yaml"))
audit = audit_dataset(data_yaml)

print(json.dumps({
    "total_images": sum(s["images"] for s in audit["splits"].values()),
    "splits": {k: f"{v['images']} images, {sum(v['instances'].values())} instances"
               for k, v in audit["splits"].items()},
    "missing_anything": any(
        v.get("missing_labels") or v.get("missing_images")
        for v in audit["splits"].values()
    )
}, indent=2))

## 4. 训练配置

**A100 40GB 优势**：可以把 batch size 从 12 提到 20-24，收敛更快。

三组实验按优先级排列：

In [ ]:
# ═══════════════════════════════════════════════
# 你要跑的每组实验配置
# 可以按需增删、改 name / epochs / batch
# ═══════════════════════════════════════════════

EXPERIMENTS = [
    {
        "name": "ccv2_yolov8x_1024_balanced_200ep",
        "model": "yolov8x-seg.pt",
        "epochs": 200,
        "batch": 20,
        "balance": True,
        "description": "E1: YOLOv8x + balanced oversampling, 200 epochs (最高优先级)",
    },
    {
        "name": "ccv2_yolo11x_1024_balanced_200ep",
        "model": "yolo11x-seg.pt",
        "epochs": 200,
        "batch": 16,
        "balance": True,
        "description": "E2: YOLO11x + balanced, 看新架构是否更好",
    },
    {
        "name": "ccv2_yolov8x_1024_baseline_200ep",
        "model": "yolov8x-seg.pt",
        "epochs": 200,
        "batch": 20,
        "balance": False,
        "description": "E3: YOLOv8x 不均衡基线, 200 epochs (消融对照)",
    },
]

print("计划训练的实验:")
for i, e in enumerate(EXPERIMENTS):
    bal = "balanced" if e["balance"] else "baseline"
    print(f"  {i+1}. {e['name']} ({e['model']}, {e['epochs']}ep, batch={e['batch']}, {bal})")

## 5. 开始训练

In [ ]:
import subprocess, time, json
from datetime import datetime

DATA_YAML = os.path.join(DATA, "data.yaml")
LOG = []

for idx, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(EXPERIMENTS)}] {exp['description']}")
    print(f"{'='*60}")
    start = time.time()
    
    cmd = [
        sys.executable, "train.py",
        "--model", exp["model"],
        "--data", DATA_YAML,
        "--device", "0",
        "--imgsz", "1024",
        "--batch", str(exp["batch"]),
        "--epochs", str(exp["epochs"]),
        "--name", exp["name"],
    ]
    if exp["balance"]:
        cmd += ["--balance", "--minority-classes", "1", "2", "--minority-copies", "3"]
    
    print("Command:", " ".join(cmd))
    
    try:
        result = subprocess.run(cmd, cwd=PROJECT, check=True)
        elapsed_m = (time.time() - start) / 60
        LOG.append({"name": exp["name"], "status": "OK", "elapsed_min": round(elapsed_m, 1)})
        print(f"✓ 完成 ({elapsed_m:.1f} min)")
    except subprocess.CalledProcessError as e:
        elapsed_m = (time.time() - start) / 60
        LOG.append({"name": exp["name"], "status": "FAILED", "elapsed_min": round(elapsed_m, 1)})
        print(f"✗ 失败 ({elapsed_m:.1f} min), 继续下一个...")

print(f"\n{'='*60}")
print("训练汇总")
print(f"{'='*60}")
for entry in LOG:
    status_icon = "✓" if entry["status"] == "OK" else "✗"
    print(f"  {status_icon} {entry['name']} ({entry['elapsed_min']} min) - {entry['status']}")

## 6. 评估最优模型

对每个成功训练的实验跑正式评估。

In [ ]:
TRAIN_DIR = os.path.join(PROJECT, "runs", "train")

for exp in EXPERIMENTS:
    best_pt = os.path.join(TRAIN_DIR, exp["name"], "weights", "best.pt")
    if not os.path.exists(best_pt):
        print(f"SKIP {exp['name']} — best.pt not found")
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {exp['name']}")
    
    for imgsz in [640, 1024]:
        output = os.path.join(PROJECT, "runs", "evaluate", f"{exp['name']}_{imgsz}")
        cmd = [
            sys.executable, "evaluate.py",
            "--weights", best_pt,
            "--data", DATA_YAML,
            "--split", "test",
            "--imgsz", str(imgsz),
            "--output", output,
            "--save-visuals",
        ]
        print(f"  imgsz={imgsz} ...")
        subprocess.run(cmd, cwd=PROJECT)
        
        # 打印关键指标
        mf = os.path.join(output, "metrics.json")
        if os.path.exists(mf):
            with open(mf) as f:
                m = json.load(f)
            print(f"    Box  mAP50: {m['map']['box']['map50']:.3f}  |  mAP50-95: {m['map']['box']['map']:.3f}")
            print(f"    Mask mAP50: {m['map']['mask']['map50']:.3f}  |  mAP50-95: {m['map']['mask']['map']:.3f}")
            print(f"    Per-image count MAE: {m['counts']['per_image_total']['mae']:.3f}")
            print(f"    Inference: {m['counts']['latency']['inference']['mean_ms']:.0f} ms")

## 7. 对比汇总表

In [ ]:
print(f"{'Experiment':<45} {'Box mAP50':>10} {'Mask mAP50':>10} {'Count MAE':>10} {'Latency':>10}")
print("-" * 90)

for exp in EXPERIMENTS:
    for imgsz in [640, 1024]:
        mf = os.path.join(PROJECT, "runs", "evaluate", f"{exp['name']}_{imgsz}", "metrics.json")
        if not os.path.exists(mf):
            continue
        with open(mf) as f:
            m = json.load(f)
        box = m['map']['box']['map50']
        mask = m['map']['mask']['map50']
        mae = m['counts']['per_image_total']['mae']
        lat = m['counts']['latency']['inference']['mean_ms']
        label = f"{exp['name'][:35]}@{imgsz}"
        print(f"{label:<45} {box:10.3f} {mask:10.3f} {mae:10.3f} {lat:8.0f} ms")

## 8. 打包下载

训练完执行这个 Cell，会在 `/root/autodl-tmp/results.tar.gz` 生成所有产物的打包文件。
然后在 AutoDL 左侧文件浏览器右键下载即可。

In [ ]:
import tarfile, glob

OUT = "/root/autodl-tmp/results.tar.gz"

with tarfile.open(OUT, "w:gz") as tar:
    # 每个实验的 best.pt
    for exp in EXPERIMENTS:
        best = os.path.join(PROJECT, "runs", "train", exp["name"], "weights", "best.pt")
        if os.path.exists(best):
            tar.add(best, arcname=f"{exp['name']}/best.pt")
        # metrics.json
        for mname in ["metrics.json", "run_config.json"]:
            mp = os.path.join(PROJECT, "runs", "train", exp["name"], mname)
            if os.path.exists(mp):
                tar.add(mp, arcname=f"{exp['name']}/{mname}")
    
    # 评估结果
    for f in glob.glob(os.path.join(PROJECT, "runs", "evaluate", "*", "metrics.json")):
        rel = os.path.relpath(f, os.path.join(PROJECT, "runs", "evaluate"))
        tar.add(f, arcname=f"evaluate/{rel}")

size_mb = os.path.getsize(OUT) / 1e6
print(f"打包完成: {OUT} ({size_mb:.1f} MB)")
print("在左侧文件浏览器找到 results.tar.gz → 右键下载")
print(f"\n下载后在本地解压: tar -xzf results.tar.gz")

---

## 备注

- **中途想停**：直接点 Jupyter 的 ■ Stop 按钮，然后重新执行第 5 步会跳过已完成的实验（需要手动注释 EXPERIMENTS 列表）
- **TensorBoard 监控**：训练过程中在 AutoDL 终端执行 `tensorboard --logdir runs/train --port 6006`，然后通过 AutoDL 的端口转发查看
- **关机提醒**：训练完**务必关机**！AutoDL 按秒计费，不关机持续扣费。Jupyter 右上角 → 关机
- **A100 batch=20 如果 OOM**：降到 16 或 12 即可